# Ch 8　Persistence 與 checkpointer：讓 agent 有記憶、可回復

> **本 notebook 完全不需要 API key**——checkpointer 是純機制，最適合動手把「狀態怎麼被存、怎麼讀回」玩透。

In [ ]:
!uv pip install -q langgraph

## 8.2–8.3　掛 checkpointer、帶 thread_id，並數出 4 個 checkpoint

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]     # 累加，待會看 bar 怎麼一路長大

def node_a(state): return {"foo": "a", "bar": ["a"]}
def node_b(state): return {"foo": "b", "bar": ["b"]}

builder = StateGraph(State)
builder.add_node(node_a); builder.add_node(node_b)
builder.add_edge(START, "node_a"); builder.add_edge("node_a", "node_b"); builder.add_edge("node_b", END)

checkpointer = InMemorySaver()                          # 開發用：存在記憶體
graph = builder.compile(checkpointer=checkpointer)      # 掛上去，persistence 就開了

config = {"configurable": {"thread_id": "1"}}           # thread_id 是這一切的主鍵，缺它什麼都存不了
print("最終 state:", graph.invoke({"foo": "", "bar": []}, config))

In [ ]:
# 每個 super-step 邊界存一個 checkpoint。這個 START->A->B->END 的圖，應該剛好 4 個。
history = list(graph.get_state_history(config))
print("checkpoint 數量:", len(history))   # 預期 4
print("（最新在最前面）每個 checkpoint 的 next 與 values：")
for snap in history:
    print(f"  next={snap.next!s:18} values={snap.values}")

## 8.4　讀懂 StateSnapshot，並定位特定 checkpoint

In [ ]:
snap = graph.get_state(config)           # 最新狀態
print("values   :", snap.values)
print("next     :", snap.next)            # () = 已完成
print("step     :", snap.metadata["step"])
print("source   :", snap.metadata["source"])

# time travel 的前置技能：精準找到「node_b 還沒跑」的那個 checkpoint
before_b = next(s for s in history if s.next == ("node_b",))
print("\n找到 node_b 執行前的 checkpoint：", before_b.values)   # {'foo': 'a', 'bar': ['a']}

## 8.5　短期記憶：同一個 thread 會記得，換 thread 就忘光

我們用一個「把使用者說的話累積起來」的小圖，模擬對話記憶（不呼叫 LLM 也能示範記憶的本質：狀態綁在 thread 上）。

In [ ]:
class ChatState(TypedDict):
    said: Annotated[list[str], add]

def remember(state): return {}   # 這個節點故意不改 said；重點是「同一 thread 的多次 invoke 會累積」

b = StateGraph(ChatState)
b.add_node("remember", remember)
b.add_edge(START, "remember"); b.add_edge("remember", END)
chat = b.compile(checkpointer=InMemorySaver())

t1 = {"configurable": {"thread_id": "user-A"}}
chat.invoke({"said": ["我叫 Kevin"]}, t1)
chat.invoke({"said": ["我喜歡爬山"]}, t1)
print("thread user-A 累積記得：", chat.get_state(t1).values["said"])

# 換一個 thread = 全新對話，互不相干
t2 = {"configurable": {"thread_id": "user-B"}}
chat.invoke({"said": ["我是別人"]}, t2)
print("thread user-B 只記得：", chat.get_state(t2).values["said"])

## 重點回顧

`compile(checkpointer=...)` + 帶 `thread_id`，就解鎖了短期記憶、HITL、time travel、fault tolerance。開發用 `InMemorySaver`，production 換 Sqlite/Postgres，圖邏輯一行不改。